In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import time
from torchvision.datasets import ImageFolder
from torchvision.transforms import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split, Subset
import json

In [ ]:
img_pth = './plant_village' # path to dataset

train_trans = transforms.Compose([
    # transforms.Grayscale(num_output_channels=1),
    transforms.Resize([64,64]), #[128, 128]
    transforms.ToTensor()
])

test_trans = transforms.Compose([
    # transforms.Grayscale(num_output_channels=1),
    transforms.Resize([64,64]), #[128, 128]
    transforms.ToTensor()
])

image_folder = ImageFolder(img_pth)

In [ ]:
generator = torch.Generator().manual_seed(47)

train_folder_len = int(0.9 * len(image_folder))
test_folder_len = len(image_folder) - train_folder_len
validation_folder_len = int(0.1 * train_folder_len)
train_folder_len -= validation_folder_len

train_folder, validation_folder, test_folder = random_split(
    image_folder,
    [train_folder_len, validation_folder_len, test_folder_len],
    generator=generator
)
print(
    f'Train: {train_folder_len}\n'
    f'Valid: {validation_folder_len}\n'
    f'Test : {test_folder_len}\n'
    f'Sum  : {train_folder_len + validation_folder_len + test_folder_len}\n'
    f'Act  : {len(image_folder)}'
)

Train: 16717
Valid: 1857
Test : 2064
Sum  : 20638
Act  : 20638


In [4]:
train_dataset = Subset(
    ImageFolder(img_pth, transform=train_trans),
    train_folder.indices
)
validation_dataset = Subset(
    ImageFolder(img_pth, transform=test_trans),
    validation_folder.indices
)
test_dataset = Subset(
    ImageFolder(img_pth, transform=test_trans),
    test_folder.indices
)

In [5]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 2
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 2
)
test_loader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 2
)

In [39]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d( 3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        # self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        # self.bn4 = nn.BatchNorm2d(256)
        
        self.flatten = nn.Flatten()
        
        self.linear1 = nn.Linear(8 * 8 * 128, 64)
        self.linear2 = nn.Linear(64, 128)
        self.linear3 = nn.Linear(128, 38)
        
        self.drop2d = nn.Dropout2d(0.2)
        # self.drop = nn.Dropout(0.3)
        self.relu = nn.ReLU()
        self.mxpool = nn.MaxPool2d(2, 2)
        # self.gap = nn.AdaptiveAvgPool2d((1,1))

    def forward(self, x):
        # CNN Layer 1
        x = self.conv1(x)
        # x = self.bn1(x)
        x = self.relu(x)
        x = self.mxpool(x)
        x = self.drop2d(x)
        # CNN Layer 2
        x = self.conv2(x)
        # x = self.bn2(x)
        x = self.relu(x)
        x = self.mxpool(x)
        x = self.drop2d(x)
        # CNN Layer 3
        x = self.conv3(x)
        # x = self.bn3(x)
        x = self.relu(x)
        x = self.mxpool(x)
        x = self.drop2d(x)
        # CNN Layer 4 & Dropout2D
        # x = self.conv4(x)
        # x = self.bn4(x)
        # x = self.relu(x)
        # x = self.drop2d(x)
        # x = self.gap(x)
        
        ### Linear
        x = self.flatten(x)

        x = self.linear1(x)
        x = self.relu(x)

        # x = self.drop(x)

        x = self.linear2(x)
        x = self.relu(x)
        x = self.linear3(x)

        return x

### Init Model

In [40]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = CNN().to(device)

total_parameters = 0
print(f'{'Name':^20} | {'Shape':^40} | {'Num':^9}')
for name, par in model.named_parameters():
    total_parameters += par.numel()
    print(f'{name:^20} | {str(par.shape):^40} | {par.numel():>9}')
print(f'Total parameters: {total_parameters}')

        Name         |                  Shape                   |    Num   
    conv1.weight     |        torch.Size([32, 3, 3, 3])         |       864
     conv1.bias      |             torch.Size([32])             |        32
    conv2.weight     |        torch.Size([64, 32, 3, 3])        |     18432
     conv2.bias      |             torch.Size([64])             |        64
    conv3.weight     |       torch.Size([128, 64, 3, 3])        |     73728
     conv3.bias      |            torch.Size([128])             |       128
     bn1.weight      |             torch.Size([32])             |        32
      bn1.bias       |             torch.Size([32])             |        32
     bn2.weight      |             torch.Size([64])             |        64
      bn2.bias       |             torch.Size([64])             |        64
     bn3.weight      |            torch.Size([128])             |       128
      bn3.bias       |            torch.Size([128])             |       128
   linear1.w

In [ ]:
# Load old model before training (if needed)
model.load_state_dict(torch.load('best_by_loss.pth'))

<All keys matched successfully>

### Init Statistics variables
- load them from 'stats.json' file if the model is pre-trained

In [42]:
train_accuracies = []
train_losses = []
validation_accuracies = []
validation_losses = []
epoch_times = []

In [26]:
with open('stats.json', mode='r') as json_file:
    tmp = json.JSONDecoder().decode(json_file.read())
    train_accuracies = tmp['train_accuracies']
    train_losses = tmp['train_losses']
    validation_accuracies = tmp['validation_accuracies']
    validation_losses = tmp['validation_losses']
    epoch_times = tmp['epoch_times']
    del tmp

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [43]:
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(),lr=1e-3, weight_decay=1e-4)

epochs = 30
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

start_time = time.time_ns()
best_loss = float('inf')
best_accuracy = 0

print(f'{'Epoch':^5} | {'Loss':^6} | {'Accuracy':8} | {'V.Loss':^6} | {'Accuracy':^8} | {'Time':^7}')
print('-'*(5+6+8+6+8+7+15))

for epoch in range(epochs):
    model.train()

    total_batches_processed = 0
    epoch_start_time = time.time_ns()
    
    train_loss = 0
    train_correct_labels = 0
    train_total_labels = 0

    validation_loss = 0
    validation_correct_labels = 0
    validation_total_labels = 0
    
    print('Processing: 0%\r', end='')
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        predictions = outputs.argmax(dim=1)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_total_labels += labels.size(0)
        train_correct_labels += (predictions == labels).sum().item()
        train_loss += loss.item()
        
        total_batches_processed += 1
        print(f'Processing: {total_batches_processed/len(train_loader)*100:3.2f}%\r', end='')
        
    train_accuracy = train_correct_labels / train_total_labels * 100
    average_train_loss = train_loss / len(train_loader)

    model.eval()
    with torch.no_grad():
        for images, labels in validation_loader:
            images = images.to(device)
            labels = labels.to(device)
        
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            predictions = outputs.argmax(dim=1)
            
            validation_total_labels += labels.size(0)
            validation_correct_labels += (predictions == labels).sum().item()
            validation_loss += loss.item()

    validation_accuracy = validation_correct_labels / validation_total_labels * 100
    average_validation_loss = validation_loss / len(validation_loader)
    epoch_total_time = (time.time_ns() - epoch_start_time) / 1e9

    if average_validation_loss <= best_loss:
      torch.save(model.state_dict(), 'best_by_loss.pth')
      best_loss = average_validation_loss
    if validation_accuracy >= best_accuracy:
      torch.save(model.state_dict(), 'best_by_accuracy.pth')
      best_accuracy = validation_accuracy

    train_losses.append(average_train_loss)
    validation_losses.append(average_validation_loss)
    train_accuracies.append(train_accuracy)
    validation_accuracies.append(validation_accuracy)
    epoch_times.append(epoch_total_time)
    
    print(f'{epoch+1:^5} | {average_train_loss:^6.4f} | {train_accuracy:>7.2f}% | {average_validation_loss:^6.4f} | {validation_accuracy:>7.2f}% | {epoch_total_time:>6.2f}s')
print(f'Total run time: {(time.time_ns() - start_time)/1e9:.2f}s')

Epoch |  Loss  | Accuracy | V.Loss | Accuracy |  Time  
-------------------------------------------------------
  1   | 2.1469 |   44.34% | 1.5157 |   68.50% |  96.42s
  2   | 1.5322 |   67.54% | 1.2581 |   78.35% |  99.09s
  3   | 1.3336 |   75.04% | 1.1840 |   80.72% |  98.88s
  4   | 1.2187 |   79.57% | 1.0452 |   87.29% | 106.45s
  5   | 1.1489 |   82.77% | 1.0834 |   84.60% |  99.71s
  6   | 1.0848 |   85.44% | 1.0244 |   87.61% |  98.44s
  7   | 1.0454 |   87.10% | 0.9784 |   88.75% |  99.57s
  8   | 1.0040 |   88.83% | 0.9462 |   90.47% | 100.98s
  9   | 0.9766 |   89.91% | 0.9018 |   92.35% |  99.35s
 10   | 0.9509 |   91.05% | 0.9277 |   90.90% | 100.00s
 11   | 0.9266 |   91.95% | 0.8942 |   93.11% | 100.03s
 12   | 0.9047 |   92.99% | 0.8778 |   93.27% |  99.27s
 13   | 0.8799 |   93.84% | 0.8969 |   92.41% |  99.81s
 14   | 0.8750 |   93.93% | 0.8613 |   93.11% | 104.84s
 15   | 0.8543 |   94.77% | 0.8739 |   92.89% | 100.01s
 16   | 0.8465 |   94.92% | 0.8487 |   93.70% | 

### Save Stats

In [44]:
x = json.JSONEncoder().encode({
    'train_accuracies': train_accuracies,
    'train_losses': train_losses,
    'validation_accuracies': validation_accuracies,
    'validation_losses': validation_losses,
    'epoch_times': epoch_times
})

with open('stats.json', mode='w') as json_file:
    if json_file.writable():
        json_file.write(x)

In [ ]:
# save the last model state (in case you want to compare all three saved models)
torch.save(model.state_dict(), 'last_train_model.pth')

In [28]:
model.eval()
correct = 0
total = 0
processed_batches = 0
print('\rProcessing: 0%', end='')
with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
        
            results = model(images)
            predictions = results.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)
            processed_batches+=1
            print(f'\rProcessing: {processed_batches/len(test_loader)*100:.2f}%', end='')

total_accuracy = float(correct)/total*100
print(f'\rAccuracy: {total_accuracy:>8.2f}%', end='')


Accuracy:    95.64%